Run once with **Accelerator: None**, Internet on. Other notebooks attach this output instead of downloading again.

In [1]:
from pathlib import Path

OUT = Path("/kaggle/working")
for d in ["ckpt", "data", "ood"]:
    (OUT / d).mkdir(exist_ok=True)

In [2]:
from huggingface_hub import hf_hub_download

for repo, filename in [
    ("nieshen/SMDM", "mdm_safetensors/mdm-170M-100e18-rsl-0.01.safetensors"),
    ("taeyoun811/whisfusion", "whisfusion_stage2_decoder.pt"),
]:
    print(hf_hub_download(repo, filename, local_dir=str(OUT / "ckpt")))

mdm_safetensors/mdm-170M-100e18-rsl-0.01(…):   0%|          | 0.00/876M [00:00<?, ?B/s]

/kaggle/working/ckpt/mdm_safetensors/mdm-170M-100e18-rsl-0.01.safetensors


whisfusion_stage2_decoder.pt:   0%|          | 0.00/1.05G [00:00<?, ?B/s]

/kaggle/working/ckpt/whisfusion_stage2_decoder.pt


In [3]:
import subprocess

# test-* to report on, dev-clean to tune on
for split in ["test-clean", "test-other", "dev-clean"]:
    target = OUT / "data" / "LibriSpeech" / split
    if target.exists():
        continue

    archive = OUT / f"{split}.tar.gz"
    subprocess.run(["curl", "-L", "--retry", "3", "-o", str(archive),
                    f"https://www.openslr.org/resources/12/{split}.tar.gz"], check=True)
    subprocess.run(["tar", "-xzf", str(archive), "-C", str(OUT / "data")], check=True)
    archive.unlink()

    print(split, len(list(target.rglob("*.flac"))))

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  330M  100  330M    0     0  5352k      0  0:01:03  0:01:03 --:--:-- 8328k


test-clean 2620


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  313M  100  313M    0     0  4660k      0  0:01:08  0:01:08 --:--:-- 4616k


test-other 2939


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  322M  100  322M    0     0  7540k      0  0:00:43  0:00:43 --:--:--  9.8M


dev-clean 2703


Out of domain: openslr SLR83, crowdsourced UK and Ireland dialects. Different accents, different recording conditions, different content. Whisfusion never saw it.

In [4]:
for name in ["midlands_english_female", "irish_english_male", "northern_english_female"]:
    target = OUT / "ood" / name
    if target.exists():
        continue

    archive = OUT / f"{name}.zip"
    subprocess.run(["curl", "-L", "--retry", "3", "-o", str(archive),
                    f"https://www.openslr.org/resources/83/{name}.zip"], check=True)
    subprocess.run(["unzip", "-q", str(archive), "-d", str(target)], check=True)
    archive.unlink()

    print(name, len(list(target.glob("*.wav"))))

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 98.3M  100 98.3M    0     0  13.3M      0  0:00:07  0:00:07 --:--:-- 17.0M


midlands_english_female 246


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  156M  100  156M    0     0  4544k      0  0:00:35  0:00:35 --:--:-- 5007k


irish_english_male 450


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  300M  100  300M    0     0  7510k      0  0:00:40  0:00:40 --:--:-- 8676k


northern_english_female 750


In [5]:
from transformers import AutoTokenizer, WhisperForConditionalGeneration, WhisperProcessor

CACHE = str(OUT / "hf")
WhisperProcessor.from_pretrained("openai/whisper-small", cache_dir=CACHE)
WhisperForConditionalGeneration.from_pretrained("openai/whisper-small", cache_dir=CACHE)
AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T",
                              cache_dir=CACHE)

preprocessor_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

TokenizersBackend(name_or_path='TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T', vocab_size=32000, model_max_length=1000000000000000019884624838656, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>'}, added_tokens_decoder={
	0: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)